# 05 — Data Transformation (pivot, reshape, strings, datetime, time series)

Maps to **MODULE-05a / 05b**.

Docs: `datasets/README.md` (Phases 3, 4 & 6)

In [ ]:
import pandas as pd

orders = pd.read_csv('datasets/raw/orders.csv', parse_dates=['order_date'])
customers = pd.read_csv('datasets/raw/customers.csv')
order_items = pd.read_csv('datasets/raw/order_items.csv')
products = pd.read_csv('datasets/raw/products.csv')

merged = (orders.merge(customers, on='customer_id')
          .merge(order_items, on='order_id')
          .merge(products, on='product_id'))

### 1. Pivot table — revenue by category × membership

In [ ]:
merged.pivot_table(
    values='line_total', index='category', columns='membership',
    aggfunc='sum', fill_value=0)

### 2. Crosstab — count by membership × source

In [ ]:
pd.crosstab(customers['membership'], customers['source'])

### 3. Wide → long (melt)

In [ ]:
wide = merged.pivot_table(values='line_total', index='category',
                              columns='membership', aggfunc='sum').reset_index()
wide.head()

In [ ]:
long = wide.melt(id_vars='category', var_name='membership', value_name='line_total')
long.head()

### 4. String operations — normalize phone numbers

In [ ]:
customers['phone'].head()   # several different formats

In [ ]:
customers['phone_clean'] = customers['phone'].str.replace(r'[^\d]', '', regex=True)
customers['phone_clean'].head()

### 5. DateTime — extract components

In [ ]:
orders['year'] = orders['order_date'].dt.year
orders['month'] = orders['order_date'].dt.month
orders['day_name'] = orders['order_date'].dt.day_name()
orders[['order_date', 'year', 'month', 'day_name']].head()

### 6. Time series — resample & rolling

In [ ]:
traffic = pd.read_csv('datasets/raw/website_traffic.csv', parse_dates=['date'])
traffic.set_index('date')['sessions'].resample('ME').sum().head()

In [ ]:
traffic.set_index('date')['sessions'].rolling(7).mean().head(20)